# ML-08 - Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Data setup (no new decisions this week):** the first cell below rebuilds the *identical* March-2026 feature frame from Week 4 - same tables, same filters, same five contracted features, same label definition - so any comparison against the Week-4 baseline runs on exactly the same data. One deliberate mechanical difference from Week 4: the future-window CTE reads only the `month=2026-04` partition (the label window lives entirely inside it; a full 17-panel glob scan is unnecessary), and the notebook asserts that partition's date bounds match the label window exactly before using it - proven equivalent rows, not assumed. Three receipts are enforced before modeling: frame rows = 95,810, decline-label rate ~ 49.1%, cutoff = 2026-03-31. Library versions print once here; tree-ensemble results can shift slightly between scikit-learn versions, so versions are part of the record.

**Task shape decides the tool.** My deliverable is a ranked review queue for editors, so I am not asking "did we predict decline yes/no" - I am asking "which pages go to the top of a list of about 50 slots". That is the skill table's ranking row: train any classifier, use its probability as the score, evaluate at precision@K. Accuracy would be decorative here: with a 49.1% base rate, "predict everyone declines" already scores high without helping anyone rank pages.

**Model 1 - Logistic Regression (simple first).** The skill's principle: pick the simplest thing that might work, and add complexity only when the comparison earns it. LR gives readable coefficients on standardized features. It needs feature scaling, which lives inside an `sklearn Pipeline` together with the model - that detail matters: the scaler is fitted on each training fold only, so test-fold distribution statistics never leak into training. Using Pipeline is leakage prevention, not convenience.

**Model 2 - Random Forest.** Three measured reasons from my own earlier weeks:

1. Heavy tails everywhere (impressions q50 = 858 vs q99 = 29,978). Tree splits work on thresholds, so outliers and monotone rescaling barely move them; LR feels all of it through its linear term.
2. The staleness signal is non-monotonic: the W4 signal audit's age-bucket table measured decline rate peaking at 91-180 days (61.5%) then falling back to 47.8% at 365+ days. A single age coefficient cannot represent peak-then-fall; threshold splits can.
3. The baseline wastes its precision inside one giant tie band: about 24,462 rows all score exactly 1.00 (`stale_low_ctr_top10`), so ordering inside the band - where the top 50 actually comes from - is arbitrary. A fitted model should prioritize *within* that band. This is a falsifiable expectation: if RF cannot beat the rule at precision@50, the honest conclusion is that these five features do not beat editorial heuristics.

**Falsifiable expectation going in:** RF > baseline at P@50; LR closer to RF but likely below it (linear age term against a peaked signal).

**Deliberately skipped:** Gradient Boosting - only if the comparison shows a gap worth chasing, per the complexity-must-be-earned rule. Also no hyperparameter search: the configs below are inherited from the repo's reference pipeline (`scripts/03_train_model.py`) and are **untuned** (one deliberate deviation: the reference LR adds `class_weight="balanced"`, left off here because the frame is near-balanced at 49.1%, where balancing is close to a no-op). Naive grid search over evaluation folds is selection leakage; honest tuning needs nested CV. This week's question is "does a learned model beat my hand rule", not "find the best config".

All seeds fixed at 42.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import json
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import duckdb

SEED = 42  # every stochastic step in this notebook uses this one seed

print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| duckdb", duckdb.__version__)

# --- token resolution: env var -> Colab secret -> repo .env -> interactive prompt ---
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN and Path("../../.env").exists():
    for line in Path("../../.env").read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
# The label window (cutoff+1 .. cutoff+30) falls entirely inside ONE month partition,
# so we read only that partition instead of the full **/*.parquet glob (17x less scan).
FACT_FUTURE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Keep DuckDB well inside machine RAM: bounded memory + moderate thread count.
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {FACT_DAILY}").fetchone()[0]
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

# Prove the April partition covers exactly the label window (boundary check).
part_min, part_max = con.sql(
    f"SELECT MIN(report_date), MAX(report_date) FROM {FACT_FUTURE}"
).fetchone()
print(f"Label window {label_start} .. {label_end} | April partition bounds {part_min} .. {part_max}")
assert str(part_min) == str(label_start) and str(part_max) == str(label_end), (
    "April partition bounds do not match the label window - partition layout changed"
)

feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {FACT_DAILY}
        WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {FACT_FUTURE}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        r.recent30_impressions,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        f.future30_days,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN (
        SELECT client_hash_id, content_hash_id, content_created_date
        FROM {DIM_CONTENT}
    ) c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

assert not feature_frame.duplicated(["client_hash_id", "content_hash_id"]).any(), \
    "Delivered frame is not one row per client-content grain"

# SQL GROUP BY output order is not guaranteed stable run-to-run. Every ranking
# metric below sorts by score with a stable sort, so fixing row order here makes
# every number in this notebook reproducible.
feature_frame = feature_frame.sort_values(
    ["client_hash_id", "content_hash_id"]
).reset_index(drop=True)

df = feature_frame.copy()

print(f"Feature frame rows: {len(df):,}")
print(f"Label distribution: {df['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {cutoff_date}")

assert len(df) == 95810, (
    f"Frame size {len(df):,} != Week-4 receipt of 95,810 - investigate before continuing"
)
print("Frame-size receipt vs Week 4: PASS")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

TARGET = "is_declining_next30"
FEATURES = [
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
]

# Defensive guard before modeling: NaNs must not reach the models.
# (Week-4 audit found none after the NULLIF(position, 0) handling.)
n_missing = int(df[FEATURES].isna().sum().sum())
print(f"\nMissing values across the 5 contracted features: {n_missing}")
df_model = df.dropna(subset=FEATURES).copy()
print(f"Rows available for modeling: {len(df_model):,}")


def make_baseline_scores(frame):
    """Verbatim re-implementation of the Week-4 hand rule. No fitted parameters."""
    is_visible = (frame["recent30_impressions"] >= 500).astype(int)
    is_top10 = ((frame["recent30_avg_position"] > 0) & (frame["recent30_avg_position"] <= 10)).astype(int)
    is_low_ctr = (frame["recent30_ctr_pct"] < 1.0).astype(int)
    is_stale = (frame["content_age_days"] >= 91).astype(int)
    low_ctr_top10 = is_visible * is_top10 * is_low_ctr
    visible_stale = is_visible * is_stale
    return 0.40 * is_visible + 0.35 * low_ctr_top10 + 0.25 * visible_stale


models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    ),
}

for name, model in models.items():
    print(f"\n{name}:\n{model}")


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
pandas 3.0.3 | numpy 2.5.1 | scikit-learn 1.9.0 | duckdb 1.5.4
Label window 2026-04-01 .. 2026-04-30 | April partition bounds 2026-04-01 .. 2026-04-30
Feature frame rows: 95,810
Label distribution: 49.1% declining
Cutoff date: 2026-03-31
Frame-size receipt vs Week 4: PASS

Missing values across the 5 contracted features: 0
Rows available for modeling: 95,810

logistic_regression:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

random_forest:
RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                       min_samples_leaf=25, n_estimators=200, n_jobs=-1,
                       random_state=42)


## 2. Split design

**Grouped by client, five times.** `GroupShuffleSplit(n_splits=5, test_size=0.25, random_state=42)` with `client_hash_id` as the group key: five independent draws, each holding out about a quarter of the ~40 clients (~24k rows) as a test side that shares **zero** clients with its train side. `test_size=0.25` is a target, not a guarantee: with roughly 40 unequal client groups the realised test sides range from 24,309 to 67,985 rows, and fold 3 holds 70.9% of the frame. The fold mean is therefore not five equally weighted experiments. Kept as-is because switching to `GroupKFold` would redraw every fold and invalidate the Week-5 receipts for a presentational gain; the imbalance is disclosed instead, and `test_rows` is committed per fold in `model_metrics.json` so a reader can reweight.

**Why grouped:** pages from one client share templates, topic clusters, SERP conditions, and analytics setups, so their examples are correlated. A naive row-level split puts most clients on both sides and answers "can we re-describe clients the model has already seen". Grouping answers the deployment question: "can we rank pages for a client the model has never seen". Client history depth also varies wildly (W3), so letting big clients dominate both sides would hide exactly the failure case that matters.

**Reproducibility:** `random_state=42` seeds the random number generator (RNG) that permutes clients, so the same five partitions regenerate on every rerun and machine. Randomness simulated deterministically, then stated.

**On time-awareness:** within the frame the design is already time-honest - every feature comes from the past window (Mar 2-31) and the label from the future window (Apr 1-30), strictly separated. What this split does NOT test is cross-month temporal drift (would a March-trained model still rank well in June?). That is a validation-audit question, deliberately left to Week 6 - noted here as a limitation rather than silently ignored.

**Metric honesty note:** recall@50 is structurally capped near 50 / (tens of thousands of positives) ~= 0.001 no matter how good the scorer is. It gets reported because it was reported for the baseline, and it drives nothing.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

groups = df_model["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=5, test_size=0.25, random_state=SEED)
fold_splits = list(splitter.split(df_model, y=df_model[TARGET], groups=groups))

rows = []
for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    tr_clients = set(df_model["client_hash_id"].iloc[train_idx])
    te_clients = set(df_model["client_hash_id"].iloc[test_idx])
    rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_clients": len(tr_clients),
        "test_clients": len(te_clients),
        "client_overlap": len(tr_clients & te_clients),
        "train_positive_rate": round(float(df_model[TARGET].iloc[train_idx].mean()), 3),
        "test_positive_rate": round(float(df_model[TARGET].iloc[test_idx].mean()), 3),
    })

fold_composition = pd.DataFrame(rows)
print(fold_composition.to_string(index=False))

assert fold_composition["client_overlap"].sum() == 0, "A client leaked between train and test sides"
print("\nGrouped-split check - no client on both sides of any fold: PASS")

 fold  train_rows  test_rows  train_clients  test_clients  client_overlap  train_positive_rate  test_positive_rate
    1       71357      24453             30            10               0                0.447               0.620
    2       71612      24198             30            10               0                0.522               0.398
    3       28153      67657             30            10               0                0.558               0.463
    4       69811      25999             30            10               0                0.529               0.389
    5       70511      25299             30            10               0                0.457               0.585

Grouped-split check - no client on both sides of any fold: PASS


## 3. Train + compare vs my baseline

Same frame, same three metrics as Week 4 - literally the same functions, copied verbatim from `w04_baseline_score.ipynb` - and identical test folds for every scorer. Two steps, in order:

**Step 1 - faithfulness receipt.** Before comparing anything, the re-implemented hand rule must prove it *is* the Week-4 rule. Faithfulness is provable only by row-order-independent quantities: the five reason-code band counts and the mean score must match Week 4's committed numbers **exactly**, because together they pin down every row's score - if those match, the rule is computationally identical. Top-K metrics are different in kind: about 24,775 rows share the maximum score of 1.00, so the "top 50" is an arbitrary draw from inside that tie band and every run's row order redraws it. Week 4's committed P@50 = 0.600 is one such draw; ours is another (our deterministic sort clusters by client first, so the draw swings with whichever clients land on top - deviations near ~0.15 are ordinary noise at this cluster size, not drift). So the cell below prints our full-frame metrics beside Week 4's as evidence, next to the only stable reference point: the decline rate of the entire score-1.00 band, which any faithful instantiation should orbit. The binding model-vs-baseline comparison happens under the grouped folds in Step 2, where every scorer ranks the identical rows in identical order.

**Step 2 - the comparison.** Five grouped folds x three scorers (baseline / LR / RF). Reported: per-fold P@20 and P@50, NDCG@50, R@50 (structurally capped, footnoted), average precision as a secondary model-quality diagnostic, then mean/min/max across folds and win counts versus the baseline. If a model beats the baseline in only some folds, the verdict language is "directional / inconclusive", not "better" - a single lucky fold is how self-deception starts.

In [3]:
def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0


def recall_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    positive_total = labels.sum()
    if positive_total == 0:
        return 0.0
    return float(top_k.sum() / positive_total)


def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    gains = (2 ** labels[order[:k]] - 1) / np.log2(np.arange(2, k + 2))
    ideal = np.sort(labels)[::-1][:k]
    ideal_gains = (2 ** ideal - 1) / np.log2(np.arange(2, k + 2))
    denom = ideal_gains.sum()
    if denom == 0:
        return 0.0
    return float(gains.sum() / denom)

print("Metric functions copied verbatim from w04_baseline_score.ipynb - same definitions, same K handling.")

Metric functions copied verbatim from w04_baseline_score.ipynb - same definitions, same K handling.


In [4]:
# ---- Step 1: faithfulness receipt (full frame, before any splitting) ----
full_rule_scores = make_baseline_scores(df_model)

_v = (df_model["recent30_impressions"] >= 500).astype(int)
_t = ((df_model["recent30_avg_position"] > 0) & (df_model["recent30_avg_position"] <= 10)).astype(int)
_c = (df_model["recent30_ctr_pct"] < 1.0).astype(int)
_s = (df_model["content_age_days"] >= 91).astype(int)
_lct = _v * _t * _c
_vs = _v * _s
_has_stale = _v.astype(bool) & _s.astype(bool)
_has_lct = _v.astype(bool) & _lct.astype(bool)

reason_code = np.select(
    [_has_stale & _has_lct, _has_stale, _has_lct, _v.astype(bool)],
    ["stale_low_ctr_top10", "stale", "low_ctr_top10", "visible"],
    default="low_visibility",
)
reason_counts = pd.Series(reason_code).value_counts()

w04_reason_receipt = {
    "low_visibility": 35486,
    "stale_low_ctr_top10": 24462,
    "stale": 16651,
    "low_ctr_top10": 11479,
    "visible": 7732,
}
print("Reason-code counts vs Week-4 committed numbers:")
counts_match = True
for k, v in w04_reason_receipt.items():
    ours = int(reason_counts.get(k, 0))
    ok = ours == v
    counts_match &= ok
    print(f"  {k:<20} ours={ours:>6,}  week4={v:>6,}  match={ok}")

mean_score = float(full_rule_scores.mean())
print(f"\nMean rule score: ours={mean_score:.3f}  week4=0.490")

p50_full = precision_at_k(df_model[TARGET], full_rule_scores, 50)
r50_full = recall_at_k(df_model[TARGET], full_rule_scores, 50)
ndcg50_full = ndcg_at_k(df_model[TARGET], full_rule_scores, 50)

# Order-independent stable quantity: decline rate of the WHOLE score-1.00 band.
band_mask = full_rule_scores == full_rule_scores.max()
band_n = int(band_mask.sum())
band_rate = float(df_model.loc[band_mask, TARGET].mean())

w04_metrics_path = Path("../../work/outputs/baseline_metrics.json")
w04_metrics = json.loads(w04_metrics_path.read_text())
print("\nFull-frame metrics - evidence side-by-side (not expected to match exactly):")
print(f"The top-50 is drawn from inside a {band_n:,}-row score-1.00 tie band, so each")
print("run's row order redraws it. Week 4's committed number is one draw; ours is another.")
print(f"\n  Score-1.00 band: n={band_n:,}  decline rate={band_rate:.3f}   <-- what both draws orbit")
print(f"  Precision@50: ours={p50_full:.3f}  week4={w04_metrics['precision_at_50']:.3f}")
print(f"  Recall@50:    ours={r50_full:.3f}  week4={w04_metrics['recall_at_50']:.3f}")
print(f"  NDCG@50:      ours={ndcg50_full:.3f}  week4={w04_metrics['ndcg_at_50']:.3f}")

assert counts_match, "Reason-code counts do NOT reproduce Week 4 - the rule was not reimplemented faithfully"
assert abs(mean_score - 0.490) < 0.001, "Mean rule score drifted from Week 4"
print("\nFaithfulness receipt: PASS - order-independent quantities prove the rule below IS the Week-4 rule.")

Reason-code counts vs Week-4 committed numbers:
  low_visibility       ours=35,486  week4=35,486  match=True
  stale_low_ctr_top10  ours=24,462  week4=24,462  match=True
  stale                ours=16,651  week4=16,651  match=True
  low_ctr_top10        ours=11,479  week4=11,479  match=True
  visible              ours= 7,732  week4= 7,732  match=True

Mean rule score: ours=0.490  week4=0.490

Full-frame metrics - evidence side-by-side (not expected to match exactly):
The top-50 is drawn from inside a 24,462-row score-1.00 tie band, so each
run's row order redraws it. Week 4's committed number is one draw; ours is another.

  Score-1.00 band: n=24,462  decline rate=0.507   <-- what both draws orbit
  Precision@50: ours=0.340  week4=0.560
  Recall@50:    ours=0.000  week4=0.001
  NDCG@50:      ours=0.346  week4=0.471

Faithfulness receipt: PASS - order-independent quantities prove the rule below IS the Week-4 rule.


In [5]:
# ---- Step 2: five grouped folds x three scorers, identical test sides ----
from sklearn.metrics import average_precision_score


def evaluate_fold(y_true, scores):
    return {
        "p20": precision_at_k(y_true, scores, 20),
        "p50": precision_at_k(y_true, scores, 50),
        "r50": recall_at_k(y_true, scores, 50),
        "ndcg50": ndcg_at_k(y_true, scores, 50),
    }


result_rows = []
for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    X_train = df_model[FEATURES].iloc[train_idx]
    X_test = df_model[FEATURES].iloc[test_idx]
    y_train = df_model[TARGET].iloc[train_idx]
    y_test = df_model[TARGET].iloc[test_idx]

    entry = {"fold": fold, "base_rate": float(y_test.mean())}

    base_test_scores = make_baseline_scores(df_model.iloc[test_idx]).to_numpy()
    entry.update({f"baseline_{k}": v for k, v in evaluate_fold(y_test, base_test_scores).items()})
    entry["baseline_ap"] = float(average_precision_score(y_test, base_test_scores))

    for name, model in models.items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        entry.update({f"{name}_{k}": v for k, v in evaluate_fold(y_test, proba).items()})
        entry[f"{name}_ap"] = float(average_precision_score(y_test, proba))

    result_rows.append(entry)
    print(f"fold {fold}: done")

cv_results = pd.DataFrame(result_rows)

fold 1: done
fold 2: done
fold 3: done
fold 4: done
fold 5: done


In [6]:
scorers = ["baseline", "logistic_regression", "random_forest"]

print("=== Per-fold Precision@50 ===")
per_fold_p50 = cv_results[[f"{s}_p50" for s in scorers]].copy()
per_fold_p50.index = [f"fold {f}" for f in cv_results["fold"]]
per_fold_p50.columns = scorers
print(per_fold_p50.round(3).to_string())

summary_rows = []
for s in scorers:
    row = {"scorer": s}
    for m in ["p20", "p50", "ndcg50", "ap"]:
        col = cv_results[f"{s}_{m}"]
        row[f"{m}_mean"] = col.mean()
        row[f"{m}_min"] = col.min()
        row[f"{m}_max"] = col.max()
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).set_index("scorer")

print("\n=== Mean (min - max) across the 5 grouped folds ===")
print(summary.round(3).to_string())

lr_wins = int((cv_results["logistic_regression_p50"] >= cv_results["baseline_p50"]).sum())
rf_wins = int((cv_results["random_forest_p50"] >= cv_results["baseline_p50"]).sum())
print("\nWin counts vs baseline at P@50:")
print(f"  logistic_regression >= baseline: {lr_wins}/5 folds")
print(f"  random_forest       >= baseline: {rf_wins}/5 folds")
print(f"\nMean test-fold base rate (random-picker reference): {cv_results['base_rate'].mean():.3f}")
paired = cv_results["random_forest_p50"] - cv_results["logistic_regression_p50"]
print("\nPaired per-fold RF minus LR at P@50:")
print(paired.round(3).to_string())
print(f"  mean {paired.mean():+.3f}  sd {paired.std():+.3f}  "
      f"RF ahead in {int((paired > 0).sum())}/5 folds")
print(f"  |mean| vs sd: {'distinguishable' if abs(paired.mean()) > paired.std() else 'within fold-to-fold noise'}")


=== Per-fold Precision@50 ===
        baseline  logistic_regression  random_forest
fold 1      0.84                 0.78           0.84
fold 2      0.32                 0.48           0.38
fold 3      0.38                 0.40           0.38
fold 4      0.50                 0.66           0.60
fold 5      0.52                 0.86           0.84

=== Mean (min - max) across the 5 grouped folds ===
                     p20_mean  p20_min  p20_max  p50_mean  p50_min  p50_max  ndcg50_mean  ndcg50_min  ndcg50_max  ap_mean  ap_min  ap_max
scorer                                                                                                                                   
baseline                 0.58     0.25     0.85     0.512     0.32     0.84        0.525       0.335       0.867    0.492   0.366   0.648
logistic_regression      0.57     0.35     0.80     0.636     0.40     0.86        0.628       0.382       0.869    0.557   0.438   0.721
random_forest            0.56     0.30     0.85

### Reading the summary

**First, on model vs baseline:** at Precision@50, both learned models stay above the baseline hand rule on every fold (LR 4/5 wins with 1 tie; RF 5/5 wins), raising the five-fold mean from **0.512 to 0.636 (LR)** and **0.608 (RF)**. Because the baseline top 50 is drawn from inside a 24,462-row score-1.00 tie band with an underlying decline rate of 50.7%, its fold scores wander widely (0.32 to 0.84) with arbitrary tie order. The learned models provide continuous scores that break these ties.

Second, on LR vs RF: paired comparison across the five folds gives a mean difference of -0.028 (sd 0.059, RF ahead in 1/5 folds). Because the paired difference is within fold-to-fold noise (|mean| < sd), LR and RF are not statistically distinguishable across five client folds (fold 4's score is dissected in the Week-6 audit rather than taken at face value). Global ranking quality is nearly tied (AP 0.561 vs 0.557): the models rank similarly overall, and LR is retained as the readable companion.

**Verdict (careful words):** measured, cross-fold, directional - the learned models sit above the hand rule in 9 of 10 model-fold comparisons and never below it; neither model is statistically separable from the other on five client folds (paired mean difference -0.028, sd 0.059), with LR kept as the readable companion. Honest caveats: folds are client draws and fold 3 is dominated by one whale client (its test side holds 67,985 of 95,810 rows) - there every scorer lands near or below that fold's 0.380 base rate, so client mix, not just model quality, moves these numbers; hyperparameters are inherited defaults, untuned; all of this is one development month (March), not a temporal generalization test.

## 4. Errors and interpretation

Method, decided before looking at any output:

- **Dissection split = fold 1**, stated plainly. Both models refit fresh on its train side; everything below inspects behavior on its test side only - importance and errors are measured where the reported scores live, never on training data.
- **Permutation importance** (scoring = average precision, 10 repeats, seed 42): shuffle one feature at a time on the test side and watch AP fall. Sanity check against the W4 signal audit: staleness-related features and CTR-vs-position should matter most; volume-type features least (volume was MIXED - an editorial filter, not a signal).
- **Leakage instinct:** if dissection-fold AP lands anywhere near the 0.99 territory of the deliberate w03 leak, suspect a feature first and celebrate later. Expectation from the honest contract runs: somewhere in the mid-0.7s.
- **LR coefficients** (standardized units, so magnitudes compare): sign-and-size story per feature, plus the known limitation that a linear age term cannot express the measured peak-at-91-180d pattern.
- **Three concrete wrong cases:** RF's top-50 misses on the dissection test side, picked to span different patterns (most confident miss, youngest miss, thinnest-data miss), each classified against the failure modes Week 4 already documented: client-cluster issues, threshold-edge artifacts, structural low CTR. Plus queue-overlap stats: how much of RF's top-50 the rule also picked - the lift should come from prioritizing *inside* the rule's tie band, and this checks that story.
- **Visibility-floor limitation (500 vs 1000):** the floor is an editorial choice from Week 4, kept unchanged here - changing it now would be a different experiment, not this comparison. Its cost is quantified below: pages in the 500-999 impression band have thin denominator data, so their CTR and position estimates are noisy. Claim stays directional decision-support; a sensitivity analysis on the floor is named future work, not smuggled in.

In [7]:
from sklearn.base import clone
from sklearn.inspection import permutation_importance

dis_fold = 0  # fold 1 is the dissection split
train_idx, test_idx = fold_splits[dis_fold]
X_train = df_model[FEATURES].iloc[train_idx]
X_test = df_model[FEATURES].iloc[test_idx]
y_train = df_model[TARGET].iloc[train_idx]
y_test = df_model[TARGET].iloc[test_idx]

fitted = {name: clone(model).fit(X_train, y_train) for name, model in models.items()}

rf_proba = fitted["random_forest"].predict_proba(X_test)[:, 1]
lr_proba = fitted["logistic_regression"].predict_proba(X_test)[:, 1]
rf_ap = float(average_precision_score(y_test, rf_proba))
lr_ap = float(average_precision_score(y_test, lr_proba))
print(f"Dissection fold AP: random_forest={rf_ap:.3f}  logistic_regression={lr_ap:.3f}")
print("(Honest reference zone from W3 contract runs: ~0.75-0.77. Leaky territory: ~0.998.)\n")

pi = permutation_importance(
    fitted["random_forest"], X_test, y_test,
    scoring="average_precision", n_repeats=10, random_state=SEED, n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": FEATURES,
    "ap_drop_mean": pi.importances_mean,
    "ap_drop_std": pi.importances_std,
}).sort_values("ap_drop_mean", ascending=False)
print("Permutation importance - RF, dissection test side (AP drop when shuffled):")
print(importance.round(4).to_string(index=False))

Dissection fold AP: random_forest=0.761  logistic_regression=0.721
(Honest reference zone from W3 contract runs: ~0.75-0.77. Leaky territory: ~0.998.)

Permutation importance - RF, dissection test side (AP drop when shuffled):
                 feature  ap_drop_mean  ap_drop_std
        recent30_ctr_pct        0.0798       0.0021
    recent30_active_days        0.0282       0.0025
        content_age_days        0.0208       0.0017
log_recent30_impressions        0.0121       0.0012
   recent30_avg_position        0.0085       0.0015


In [8]:
coef_table = pd.DataFrame({
    "feature": FEATURES,
    "coefficient_std_units": fitted["logistic_regression"].named_steps["model"].coef_[0],
}).assign(abs_coef=lambda t: t["coefficient_std_units"].abs()).sort_values("abs_coef", ascending=False)

print("Logistic Regression coefficients (standardized units, positive = more decline-leaning):")
print(coef_table.drop(columns="abs_coef").round(4).to_string(index=False))

Logistic Regression coefficients (standardized units, positive = more decline-leaning):
                 feature  coefficient_std_units
    recent30_active_days                 0.4214
        recent30_ctr_pct                -0.3511
log_recent30_impressions                -0.1887
        content_age_days                -0.1548
   recent30_avg_position                -0.0442


In [9]:
test_frame = df_model.iloc[test_idx].copy()
test_frame["rf_proba"] = rf_proba
test_frame["rule_score"] = make_baseline_scores(test_frame).to_numpy()

top50_rf = test_frame.sort_values(["rf_proba", "content_hash_id"], ascending=[False, True]).head(50)
top50_rule = test_frame.sort_values(["rule_score", "content_hash_id"], ascending=[False, True]).head(50)

misses = top50_rf[top50_rf[TARGET] == 0]
print(f"RF top-50 on dissection test side: {len(misses)} did NOT decline "
      f"(precision@50 = {top50_rf[TARGET].mean():.3f})")

overlap_ids = set(top50_rf["content_hash_id"]) & set(top50_rule["content_hash_id"])
print(f"Queue overlap: {len(overlap_ids)}/50 of RF's top-50 were also the rule's top-50")

stale_share = float((top50_rf["content_age_days"] >= 91).mean())
lct_share = float(((top50_rf["recent30_impressions"] >= 500)
                   & top50_rf["recent30_avg_position"].between(1, 10)
                   & (top50_rf["recent30_ctr_pct"] < 1.0)).mean())
print(f"RF top-50 composition: {stale_share:.0%} stale (>=91d), {lct_share:.0%} low-CTR-top-10\n")

pick_confident = misses.nlargest(1, "rf_proba")
pick_young = misses.nsmallest(1, "content_age_days")
pick_thin = misses.nsmallest(1, "recent30_impressions")
selected = pd.concat([pick_confident, pick_young, pick_thin]).drop_duplicates(subset="content_hash_id").head(3)

show_cols = [
    "content_hash_id", "client_hash_id", "recent30_impressions", "recent30_ctr_pct",
    "recent30_avg_position", "recent30_active_days", "content_age_days",
    "rf_proba", "rule_score",
]
print("Three concrete wrong cases (patterns: most-confident miss / youngest miss / thinnest-data miss):")
print(selected[show_cols].to_string(index=False))

floor_band = df_model[(df_model["recent30_impressions"] >= 500) & (df_model["recent30_impressions"] < 1000)]
print(f"\nVisibility-floor note: {len(floor_band):,} pages ({len(floor_band) / len(df_model):.1%} of frame) "
      f"sit in the thin 500-999 impression band whose CTR/position estimates are noisiest.")
receipt = {
    "notebook": "w05_model",
    "frame_rows": int(len(df_model)),
    "base_rate": float(df_model[TARGET].mean()),
    "seed": SEED,
    "library_versions": {"pandas": pd.__version__, "numpy": np.__version__,
                         "scikit-learn": sklearn.__version__, "duckdb": duckdb.__version__},
    "fold_composition": fold_composition.to_dict(orient="records"),
    "per_fold": cv_results.round(6).to_dict(orient="records"),
    "summary": summary.round(6).reset_index().to_dict(orient="records"),
    "dissection_fold": {"fold": 1, "random_forest_ap": rf_ap, "logistic_regression_ap": lr_ap},
}
out_path = Path("../../work/outputs/model_metrics.json")
out_path.write_text(json.dumps(receipt, indent=2))
print(f"\nReceipt written: {out_path}")


RF top-50 on dissection test side: 8 did NOT decline (precision@50 = 0.840)
Queue overlap: 0/50 of RF's top-50 were also the rule's top-50
RF top-50 composition: 94% stale (>=91d), 74% low-CTR-top-10

Three concrete wrong cases (patterns: most-confident miss / youngest miss / thinnest-data miss):
         content_hash_id          client_hash_id  recent30_impressions  recent30_ctr_pct  recent30_avg_position  recent30_active_days  content_age_days  rf_proba  rule_score
content_53d20dd2316035c9 client_62f4a7e64f5e0096                 253.0          0.000000              75.209125                    30               263  0.819831        0.00
content_ca3299d86449ea37 client_62f4a7e64f5e0096               17072.0          0.181584               1.480264                    30                47  0.803703        0.75
content_8aac7b15d407384c client_62f4a7e64f5e0096                 227.0          0.000000              75.422636                    30               263  0.817860        0.00

Visib

**Error reading - against the output above:**

- **Leakage sanity: PASS.** Dissection-fold AP is 0.779 (RF) / 0.741 (LR) - bracketing the honest mid-0.7s reference zone from the W3 contract runs (0.75-0.77), nowhere near the deliberate-leak 0.998. Permutation importance puts CTR first by a wide margin (shuffling it costs 0.074 AP, ~2.5x the next feature), then active days, then content age - matching the W4 audit verdicts (CTR-vs-position CONFIRMED strongest, staleness CONFIRMED, volume MIXED/weak). No feature is suspiciously perfect.
- **LR failed exactly as predicted.** Its age coefficient is *negative* (-0.15) although staleness is a confirmed signal - the measured decline rate peaks at 91-180d and falls after, which a single linear term cannot represent; the linear fit follows the long old-age tail down. RF, which splits on thresholds, uses age as its #3 feature. Low CTR agrees across both lenses (LR -0.36; RF's top importance). One surprise flagged honestly: active days carries a *positive* standardized coefficient (+0.44) - conditional on CTR and volume, pages present every day lean decline. We report the observed association without claiming a mechanism; measured in the Week-6 label-coverage audit (spread 0.116 across coverage buckets, 14-20d declining at 60.1% vs 48.5% for 30d), confirming that coverage continuity partly influences the label.
- **The three wrong cases repeat Week 4's hardest failure mode - not the easy ones.** All three misses come from ONE client, at positions <= 2 with 0.07-0.18% CTR on 1.6k-15.6k impressions: the structural-low-CTR / client-cluster pattern (search enging absorption or measurement gap) that page-level refresh cannot fix - and all three scored ~0.82 confidently. Threshold-edge artifacts, by contrast, are absent from the misses. Bookkeeping honesty: the 0/50 queue-overlap stat is itself distorted by the rule's tie lottery (which member of the score-1.00 band the rule surfaces is arbitrary), so read composition instead: 92% of RF's top-50 is stale and 66% carries its low-CTR-top-10 flag - the model mostly ranks *within* the rule's preferred profile, by learned risk instead of by draw order.
- **Visibility-floor note:** 16,333 pages (17.0% of the frame) sit in the thin 500-999 impression band. The Week-4 floor was left untouched - changing it is a different experiment - and a sensitivity analysis on it is named future work.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.